In [1]:
# 1. Instalacja wymaganego narzędzia do dekompresji (naprawa błędu Colaba)
!apt-get update && apt-get install -y zstd

# 2. Instalacja Ollama i bibliotek Pythona
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama pymupdf sentence-transformers faiss-cpu

import subprocess
import time

# 3. Uruchomienie serwera Ollama w tle maszyny wirtualnej
print("\nUruchamianie lokalnego serwera Ollama...")
subprocess.Popen(["ollama", "serve"])
time.sleep(5) # Czekamy 5 sekund na pełny start serwera

# 4. Pobranie modelu (Llama 3.2)
print("Pobieranie modelu llama3.2 (to potrwa chwilę)...")
!ollama pull llama3.2
print("Gotowe! Silnik LLM jest gotowy do pracy.")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Get:12 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InReleas

In [2]:
import os
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm
import ollama
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")

print("Ładowanie modelu wektoryzującego...")
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())
metadata = []

class AdminRAG:
    def __init__(self, embedder, index, metadata, chunk_size=400):
        self.embedder = embedder
        self.index = index
        self.metadata = metadata
        self.chunk_size = chunk_size

    def process_file(self, file_path):
        text_data = []

        # Obsługa plików TXT
        if file_path.endswith('.txt'):
            with open(file_path, 'r', encoding='utf-8') as f:
                text_data.append((0, f.read().replace('\n', ' ')))

        # Obsługa plików PDF (opcjonalnie)
        elif file_path.endswith('.pdf'):
            pdf_document = pymupdf.open(file_path)
            for page_num in range(len(pdf_document)):
                page = pdf_document.load_page(page_num)
                text_data.append((page_num, str(page.get_text()).replace("\n", " ")))
        else:
            return

        chunks = []
        for page_num, page_text in text_data:
            page_chunks = [(page_num, page_text[i:i+self.chunk_size]) for i in range(0, len(page_text), self.chunk_size)]
            chunks.extend(page_chunks)

        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc=f"Dodawanie: {os.path.basename(file_path)}")):
            embeddings = self.embedder.encode(chunk, show_progress_bar=False)
            self.index.add(np.array([embeddings]))
            self.metadata.append({
                "filename": os.path.basename(file_path),
                "page_number": page_number,
                "chunk": chunk
            })

    def answer_question(self, query):
        q_embedding = self.embedder.encode(query, show_progress_bar=False)
        D, I = self.index.search(np.array([q_embedding]), 4)
        chunks = [self.metadata[i] for i in I[0]]

        context = ""
        for i, chunk in enumerate(chunks):
            context += f"Dokument [{i+1}]: {chunk['chunk']}\n"

        # Restrykcyjny prompt systemowy dla Administratora IT
        messages = [
            {
                "role": "system",
                "content": (
                    "Jesteś zaawansowanym asystentem wspierającym administrację systemami operacyjnymi i sieciami komputerowymi. "
                    "Twoim zadaniem jest pomaganie inżynierom poprzez precyzyjne wskazywanie komend i konfiguracji. "
                    "Odpowiadaj w języku polskim TYLKO i wyłącznie na podstawie dostarczonej dokumentacji z kontekstu. "
                    "Jeśli w kontekście brakuje rozwiązania, napisz DOKŁADNIE: "
                    "'Brak informacji w wczytanej dokumentacji systemowej.' "
                    "Kategorycznie zabraniam zmyślania komend lub wymyślania konfiguracji, których nie ma w plikach."
                )
            },
            {"role": "user", "content": f"Wczytana dokumentacja:\n{context}\n\nProblem zgłoszony przez inżyniera: {query}"}
        ]

        response = ollama.chat(model="llama3.2", messages=messages, options={"temperature": 0.1})
        return response['message']['content'], chunks

rag = AdminRAG(embedder, index, metadata)

Ładowanie modelu wektoryzującego...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
from IPython.display import display, Markdown
import os

#Ścieżka wskazująca na folder knowledge wewnątrz sample_data
knowledge_dir = "knowledge/"

# Sprawdzenie czy folder istnieje
if not os.path.exists(knowledge_dir):
    print(f"Błąd: Nie znaleziono folderu pod ścieżką '{knowledge_dir}'. Sprawdź strukturę plików po lewej stronie!")
else:
    # Wczytywanie plików z folderu do bazy wektorowej FAISS
    print("Inicjalizacja bazy wiedzy...\n")
    for file in os.listdir(knowledge_dir):
        if file.endswith('.txt') or file.endswith('.pdf'):
            rag.process_file(os.path.join(knowledge_dir, file))

    print(f"\nGotowe! Liczba wektorów w bazie FAISS: {index.ntotal}\n")

    # Scenariusze testowe do weryfikacji
    pytania = [
        "Jak zabezpieczyć porty dostępowe na przełączniku przed atakami MAC Flooding?",
        "W jaki sposób usunąć osierocone pakiety w systemie Arch Linux używając pacmana?",
        "Podaj komendę na wykonanie kopii zapasowej bazy PostgreSQL 15, która działa w kontenerze Docker.",
        "Jak skonfigurować routing BGP w naszej sieci?" # Pytanie testujące odporność (BGP nie ma w dokumentach)
    ]

    # Odpytywanie asystenta Admin-RAG
    for pyt in pytania:
        odpowiedz, zrodla = rag.answer_question(pyt)

        display(Markdown(f"### **Pytanie Inżyniera:** {pyt}"))
        display(Markdown(f"**Odpowiedź Admin-RAG:**\n\n{odpowiedz}"))
        display(Markdown("**Znalezione fragmenty dokumentacji:**"))
        for i, chunk in enumerate(zrodla):
            fragment = chunk['chunk'][:150].strip() + "..."
            display(Markdown(f"**[{i+1}]** Plik: `{chunk['filename']}`\n> _{fragment}_"))
        display(Markdown("<hr>"))

Inicjalizacja bazy wiedzy...



Dodawanie: procedury_cisco_routing.txt: 100%|██████████| 4/4 [00:01<00:00,  2.08it/s]



Gotowe! Liczba wektorów w bazie FAISS: 12



### **Pytanie Inżyniera:** Jak zabezpieczyć porty dostępowe na przełączniku przed atakami MAC Flooding?

**Odpowiedź Admin-RAG:**

TYLKO. Aby zabezpieczyć porty dostępowe na przełączniku przed atakami MAC Flooding, należy wykorzystać poniższe komendy:

1. Ustawienie trybu access na port:
   ```bash
switchport mode access
```

2. Konfiguracja portu security:
   ```bash
switchport port-security
```
   Następnie ustawiamy maksymalną liczbę adresów MAC na port:
   ```bash
switchport port-security maximum 2
```

3. Definiowanie akcji w przypadku naruszenia (violation):
   ```bash
switchport port-security violation shutdown
```
   Ponadto, należy ustawić, aby adresy MAC były trwałe:
   ```bash
switchport port-security mac-address sticky
```

Po wykonalu powyższych kroków, przełącznik powinien być zabezpieczony przed atakami MAC Flooding.

...

**[1]** Plik: `procedury_cisco_routing.txt`
> _DOKUMENTACJA SIECIOWA: ROUTING I SWITCHING (CISCO) Wersja: 1.2 Dotyczy: Infrastruktura brzegowa i szkieletowa  1. Konfiguracja OSPF (Open Shortest Pat..._

**[2]** Plik: `procedury_cisco_routing.txt`
> _ID: `router-id 10.0.0.1` - Deklaracja uczestnictwa sieci: `network 192.168.10.0 0.0.0.255 area 0` Uwaga: Koszt (cost) interfejsu dla łączy GigabitEth..._

**[3]** Plik: `procedury_cisco_routing.txt`
> _e sobą (uplinki) muszą być skonfigurowane jako 802.1Q trunk. Należy wyłączyć negocjację DTP komendą `switchport nonegotiate`. Natywny VLAN (VLAN niena..._

**[4]** Plik: `linux_arch_administracja.txt`
> _ka domyślna: Blokowanie wszystkiego na wejściu (incoming), zezwalanie na wyjście (outgoing). Komendy wdrożeniowe: `sudo ufw default deny incoming` `su..._

<hr>